# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 30 |
| **Integrantes** | Andres Felipe Grisales Vahos · Hanna Jineth Contreras Salinas · Arley Steven Franco Galeano |
| **Caso de estudio** | Wanderbricks |
| **Fecha de entrega** | Domingo 27 de septiembre |

> a la propia. Un enlace inaccesible se califica como no entregado.

**Preparación del entorno**

Esta celda deja la sesión apuntando al catálogo y al esquema que creamos en la EA1 (`bigdata_grupo30.wanderbricks`), importa las funciones de PySpark y los módulos que usa la función de medición de la sección 3, y lista las tablas disponibles para verificar que estamos trabajando sobre el mismo entorno de las evidencias anteriores y no sobre una copia distinta.

In [0]:
from pyspark.sql import functions as F
import time, statistics

CATALOGO = "bigdata_grupo30"
ESQUEMA  = "wanderbricks"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

print(f"Trabajando en {CATALOGO}.{ESQUEMA}\n")
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

# 1 · Contexto

Wanderbricks opera un marketplace de alquiler vacacional en el que los usuarios navegan propiedades desde distintos dispositivos y canales de origen antes de decidir reservar. Desde la EA1, el equipo de marketing quiere saber qué combinaciones de dispositivo y canal se traducen efectivamente en reservas confirmadas y pagadas, y cuáles solo generan tráfico sin conversión, porque de esa respuesta depende en qué canales conviene invertir.

En la **EA1** modelamos ese caso sobre un lakehouse: creamos el catálogo `bigdata_grupo30`, construimos las capas bronce (dato crudo con metadatos de ingesta) y plata (dato limpio, tipado y con el struct `metadata` de `clickstream` ya aplanado en `device` y `referrer`), demostramos las cuatro garantías de Delta Lake y respondimos las preguntas de negocio con cinco consultas analíticas. En la **EA2** nos ocupamos del entorno donde esos datos viven: organizamos el esquema por capas, aplicamos permisos diferenciados con Unity Catalog, documentamos el linaje y automatizamos el procesamiento con un Job de dos tareas encadenadas, además de comparar el modelo PaaS que usamos frente a un equivalente en IaaS.

Este proyecto cierra ese recorrido en tres frentes. Primero, llevamos el pipeline a su capa de consumo con una tabla oro que responde la pregunta de negocio central de la EA1: **¿qué combinaciones de dispositivo y canal de origen convierten mejor de navegación a reserva confirmada y pagada?** Segundo, medimos y optimizamos la operación distribuida que sostiene ese resultado el cruce entre la navegación y las reservas, documentando el plan de ejecución antes y después. Y tercero, respondemos la pregunta orientadora del curso apoyándonos en lo que construimos, no en generalidades.

# 2 · Capa oro

## 2.1 · Qué pregunta responde

**¿Qué combinaciones de dispositivo y canal de origen convierten mejor de navegación a reserva confirmada y pagada?**

Esta tabla está dirigida al equipo de marketing de Wanderbricks, que necesita decidir en qué canales concentrar la inversión. Para que le sirva sin intermediación técnica, cumple tres condiciones. La primera es que no contiene identificadores: cada fila se describe con el nombre del dispositivo y del canal de origen, no con llaves internas. La segunda es que la métrica de decisión, la tasa de conversión, viene ya calculada, de modo que no hay que dividir columnas para interpretarla. Y la tercera es que el resultado se presenta ordenado de mayor a menor conversión, para que las combinaciones más efectivas queden arriba.

La tabla se construye a partir de las tres tablas de la capa plata que intervienen en el recorrido completo del usuario: `clickstream_plata` aporta la navegación con el dispositivo y el canal, `bookings_plata` las reservas y `payments_plata` la confirmación del pago. Solo se cuenta como conversión el usuario que llegó hasta el final de ese recorrido, es decir, aquel cuya reserva quedó en estado confirmado y cuyo pago quedó completado.

In [0]:
# --- Construcción de la capa oro

clickstream = spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_plata")
bookings    = spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_plata")
payments    = spark.table(f"{CATALOGO}.{ESQUEMA}.payments_plata")

# 1. Usuarios únicos que navegaron, por dispositivo y canal

navegacion = (
    clickstream
    .select("user_id", "device", "referrer")
    .dropDuplicates(["user_id", "device", "referrer"])
)

# 2. Usuarios con reserva confirmada y pago completado

reservas_confirmadas_pagadas = (
    bookings.alias("b")
    .join(
        payments.alias("p"),
        F.col("b.booking_id") == F.col("p.booking_id"),
        "inner"
    )
    .filter(
        (F.col("b.status") == "confirmed") &
        (F.col("p.status") == "completed")
    )
    .select(F.col("b.user_id").alias("user_id"))
    .dropDuplicates(["user_id"])
    .withColumn("reserva_confirmada_pagada", F.lit(1))
)

# 3. Construcción de la tabla oro

oro = (
    navegacion.alias("n")
    .join(
        reservas_confirmadas_pagadas.alias("r"),
        F.col("n.user_id") == F.col("r.user_id"),
        "left"
    )
    .groupBy(
        F.col("n.device").alias("dispositivo"),
        F.col("n.referrer").alias("canal_origen")
    )
    .agg(
        F.countDistinct(F.col("n.user_id")).alias("usuarios_navegaron"),
        F.countDistinct(
            F.when(
                F.col("r.reserva_confirmada_pagada") == 1,
                F.col("n.user_id")
            )
        ).alias("usuarios_reserva_confirmada_pagada")
    )
    .withColumn(
        "tasa_conversion_pct",
        F.round(
            F.col("usuarios_reserva_confirmada_pagada") /
            F.col("usuarios_navegaron") * 100,
            2
        )
    )
    .orderBy(F.col("tasa_conversion_pct").desc())
)

# 4. Guardar la tabla oro

oro.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.conversion_oro")

print("Tabla conversion_oro creada correctamente.")

**Interpretación**

La tabla se construye en tres pasos que corresponden a tres decisiones de análisis.

El primero define qué cuenta como "haber navegado". Como un mismo usuario genera muchos eventos en `clickstream_plata`, contar eventos sobreestimaría el alcance de cada canal; por eso se eliminan los duplicados por la combinación de usuario, dispositivo y canal. Un usuario que entró desde el celular por correo electrónico y desde el computador por búsqueda orgánica cuenta una vez en cada combinación, que es exactamente lo que el equipo de marketing necesita comparar.

El segundo define qué cuenta como "haber convertido". No basta con que exista una reserva: se exige que esté en estado confirmado y que tenga un pago completado asociado, cruzando `bookings_plata` con `payments_plata`. Esta es la diferencia entre medir intención y medir ingreso efectivo, y es la razón por la que la conversión de esta tabla es más estricta que un simple conteo de reservas.

El tercero es el tipo de cruce. Se usa un `left join` desde la navegación hacia las conversiones, y no un cruce interno, porque las combinaciones de dispositivo y canal que **no** produjeron ninguna reserva también son información de negocio: si se perdieran, la tabla mostraría solo los canales exitosos y la tasa de conversión dejaría de ser comparable entre ellos. El `countDistinct` condicionado cuenta únicamente a los usuarios que encontraron pareja en el cruce, de modo que las combinaciones sin conversión quedan en cero en lugar de desaparecer.

El resultado se persiste como tabla Delta con `overwrite`, de manera que la capa oro pueda reconstruirse cada vez que el pipeline se ejecute sin acumular versiones duplicadas del mismo cálculo.

## 2.2 · La tabla resultante

In [0]:
display(
    spark.table(f"{CATALOGO}.{ESQUEMA}.conversion_oro")
        .orderBy(F.col("tasa_conversion_pct").desc())
)

**Interpretación**

El `orderBy` se repite aquí de forma deliberada. Delta Lake no conserva el orden de las filas al escribir una tabla: los datos se distribuyen en archivos y la lectura posterior no garantiza la secuencia en que se guardaron. Ordenar en el momento de la consulta es lo que asegura que quien abra la tabla vea arriba las combinaciones de mayor conversión, que es uno de los requisitos de una capa de consumo.

In [0]:
# Documentación de la tabla y sus columnas

spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.{ESQUEMA}.conversion_oro IS
'Tabla oro de conversión de Wanderbricks. Resume, por dispositivo y canal de origen, cuántos usuarios navegaron, cuántos completaron una reserva confirmada y pagada, y la tasa de conversión entre ambos. Una fila por combinación de dispositivo y canal.'
""")

comentarios = {
    "dispositivo": "Dispositivo desde el que navegó el usuario: mobile, desktop o tablet.",
    "canal_origen": "Canal de origen por el que llegó el usuario al sitio.",
    "usuarios_navegaron": "Usuarios únicos que navegaron desde esa combinación de dispositivo y canal.",
    "usuarios_reserva_confirmada_pagada": "Usuarios únicos de esa combinación que completaron al menos una reserva confirmada y con pago completado.",
    "tasa_conversion_pct": "Porcentaje de usuarios que navegaron y terminaron con una reserva confirmada y pagada."
}

for columna, texto in comentarios.items():
    spark.sql(f"""
        ALTER TABLE {CATALOGO}.{ESQUEMA}.conversion_oro
        ALTER COLUMN {columna} COMMENT '{texto}'
    """)

display(spark.sql(f"DESCRIBE TABLE {CATALOGO}.{ESQUEMA}.conversion_oro"))

**Interpretación**

Los comentarios no son decoración: son metadatos que quedan almacenados en Unity Catalog junto con la tabla y que se muestran en el Catalog Explorer, en los tableros y en Genie. Están escritos en lenguaje de negocio y no técnico, porque su destinatario es quien consulta la tabla, no quien la construyó. El `DESCRIBE TABLE` final deja visible que cada columna quedó efectivamente documentada.

Recorrer las columnas con un diccionario, en lugar de repetir cinco sentencias `ALTER TABLE` casi idénticas, evita el error más probable en este paso: que el nombre de una columna en la documentación no coincida con el de la tabla.

## 2.3 · El recorrido del dato



In [0]:
#Recorrido del dato bronce - plata - oro

tablas_recorrido = [
    ("bookings_bronce", "Bronce"),
    ("bookings_plata", "Plata"),
    ("clickstream_bronce", "Bronce"),
    ("clickstream_plata", "Plata"),
    ("payments_bronce", "Bronce"),
    ("payments_plata", "Plata"),
    ("reviews_bronce", "Bronce"),
    ("reviews_plata", "Plata"),
    ("conversion_oro", "Oro")
]

resultados = []

for tabla, capa in tablas_recorrido:
    cantidad = spark.table(f"{CATALOGO}.{ESQUEMA}.{tabla}").count()
    
    resultados.append(
        (capa, tabla, cantidad)
    )

df_recorrido = spark.createDataFrame(
    resultados,
    ["capa", "tabla", "cantidad_registros"]
)

display(
    df_recorrido.orderBy(
        F.when(F.col("capa") == "Bronce", 1)
         .when(F.col("capa") == "Plata", 2)
         .otherwise(3),
        "tabla"
    )
)

In [0]:
#Diferenfcia entre bronce y plata

pares = [
    ("bookings_bronce", "bookings_plata"),
    ("clickstream_bronce", "clickstream_plata"),
    ("payments_bronce", "payments_plata"),
    ("reviews_bronce", "reviews_plata")
]

diferencias = []

for bronce, plata in pares:
    cantidad_bronce = spark.table(
        f"{CATALOGO}.{ESQUEMA}.{bronce}"
    ).count()
    
    cantidad_plata = spark.table(
        f"{CATALOGO}.{ESQUEMA}.{plata}"
    ).count()
    
    diferencias.append(
        (
            bronce,
            cantidad_bronce,
            plata,
            cantidad_plata,
            cantidad_bronce - cantidad_plata
        )
    )

df_diferencias = spark.createDataFrame(
    diferencias,
    [
        "tabla_bronce",
        "registros_bronce",
        "tabla_plata",
        "registros_plata",
        "diferencia"
    ]
)

display(df_diferencias)

In [0]:
# Verificación del origen de las diferencias entre bronce y plata

display(spark.sql(f"""
    SELECT 'reviews' AS tabla,
           COUNT(*) AS filas_bronce,
           COUNT(DISTINCT review_id) AS ids_distintos,
           SUM(CASE WHEN is_deleted THEN 1 ELSE 0 END) AS marcadas_borradas
    FROM {CATALOGO}.{ESQUEMA}.reviews_bronce
    UNION ALL
    SELECT 'payments',
           COUNT(*),
           COUNT(DISTINCT payment_id),
           SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END)
    FROM {CATALOGO}.{ESQUEMA}.payments_bronce
"""))

**Interpretación**

Esta consulta no agrega información nueva al recorrido del dato: sirve para poder afirmar con evidencia a qué se debe cada diferencia, en lugar de atribuirla de memoria a las reglas de limpieza que escribimos en la EA1. Compara, para las dos tablas que sí pierden registros, el total de filas contra el número de identificadores distintos y contra la cantidad de registros marcados como borrados. El contraste entre esas tres cifras es lo que permite distinguir un descarte por duplicación de uno por calidad del dato, que son problemas distintos y se corrigen de forma distinta.

**Interpretación**

La capa bronce conserva 321.678 registros repartidos en las cuatro tablas del caso, exactamente los mismos volúmenes que documentamos en la EA1: 72.247 reservas, 100.000 eventos de navegación, 49.638 pagos y 99.793 reseñas. Vale la pena señalar que `bookings_bronce` vuelve a mostrar 72.247 filas y no las 72.250 que llegó a tener durante las pruebas de evolución de esquema de la EA1: el Job automatizado de la EA2 reescribe la capa bronce con `overwrite` en cada ejecución, de modo que la tabla regresa siempre al estado fiel del origen. Esa es precisamente la función de la capa bronce, y el conteo lo confirma.

El paso de bronce a plata no afecta a todas las tablas por igual, porque cada una recibió las reglas de limpieza que su contenido exigía.

**`bookings` y `clickstream` no pierden ningún registro.** En reservas, la diferencia es cero porque no hay identificadores duplicados ni montos nulos que descartar; es el mismo hallazgo que obtuvimos en la EA1, cuando el `DELETE` sobre montos nulos no eliminó ninguna fila. En navegación, la transformación fue estructural y no de filtrado: se extrajeron `device` y `referrer` del struct anidado `metadata` y se dejaron como columnas de primer nivel, una operación que cambia la forma del registro pero no su cantidad.

**`payments` pierde 4.465 registros, cerca del 9% de la tabla.** La verificación muestra que la causa es una sola: la tabla contiene 49.638 filas pero únicamente 45.173 `payment_id` distintos, y no hay ningún monto nulo. La diferencia corresponde por completo a pagos duplicados, que la capa plata elimina conservando una fila por identificador. La decisión es deliberada, porque un pago repetido inflaría los ingresos atribuidos a un canal y distorsionaría cualquier conclusión sobre su rentabilidad.

**`reviews` es el caso más llamativo: pasa de 99.793 registros a 1.000.** La verificación permite atribuir el recorte con precisión. Las reseñas marcadas como borradas son apenas 476, de modo que el filtro sobre `is_deleted` explica una parte mínima de la diferencia. Lo que produce el recorte es la deduplicación: la tabla de origen contiene solo **1.000 `review_id` distintos** repartidos en 99.793 filas, es decir, cada reseña aparece repetida cerca de cien veces. La capa plata conserva una fila por identificador y, al hacerlo, revela que el volumen aparente de la tabla en bronce no correspondía a información nueva sino a repetición del mismo contenido.

Este hallazgo tiene una lectura que va más allá del conteo: si hubiéramos calculado cualquier métrica de satisfacción directamente sobre la capa bronce, habríamos multiplicado por cien el peso de cada reseña sin notarlo. Es un ejemplo concreto de por qué la capa plata existe. Conviene precisar, además, que esta tabla no participa en la capa oro de este proyecto, que se construye únicamente con navegación, reservas y pagos, de modo que el recorte no incide en la tasa de conversión reportada.

**La capa oro tiene 12 filas, y esa diferencia es de naturaleza distinta a las anteriores.** No corresponde a un descarte sino a una agregación: las 12 filas son todas las combinaciones posibles entre tres dispositivos y cuatro canales de origen, y cada una resume el comportamiento de miles de usuarios. Detrás de ellas hay 96.839 combinaciones de usuario, dispositivo y canal provenientes de la navegación, de las cuales 10.874 corresponden a usuarios que completaron una reserva confirmada y pagada, es decir, un 11.23% global. Ninguna fila se perdió en este paso: lo que ocurrió fue que un volumen de cientos de miles de eventos se condensó en una tabla que cabe en una pantalla, que es exactamente el propósito de una capa de consumo.

# 3 · Medición inicial 

## 3.1 · La operación que vamos a optimizar

Elegimos el **cruce entre `clickstream_plata` y `bookings_plata` por `user_id`**, agregado por dispositivo y canal de origen. Es la operación que sostiene la capa oro de la sección 2: para saber qué proporción de los usuarios que navegaron desde un dispositivo y canal terminó reservando, hay que poner en la misma fila la navegación y las reservas de cada usuario.

Es costosa por dos razones concretas. La primera es que ninguna de las dos tablas es una dimensión pequeña: `clickstream_plata` tiene 100.000 filas y `bookings_plata` 72.247, de modo que Spark reparte **las dos** entre las máquinas del clúster para que las filas con el mismo `user_id` queden juntas. La segunda, y la que más pesa, es que la relación entre ambas tablas es **de muchos a muchos**: un mismo usuario tiene varios eventos de navegación y puede tener varias reservas, así que el cruce no conserva el volumen de entrada sino que lo multiplica, y esas filas intermedias son las que después hay que agregar.

Medimos deliberadamente la versión **sin** el `DISTINCT` previo que usamos en la EA1. Ese `DISTINCT` era una decisión analítica, porque la tasa de conversión se calcula sobre usuarios únicos y no sobre eventos; aquí nos interesa medir el cruce en su forma más exigente, que es la que permite ver el costo del movimiento de datos en el plan de ejecución.

Para poder comparar, la línea base declara la estrategia de cruce de forma explícita. Con su configuración por defecto, Spark detecta que `bookings_plata` es lo bastante pequeña para difundirla a cada máquina y aplica esa mejora por su cuenta, de modo que no existiría un caso base contra el cual medir. El laboratorio guiado resuelve esto desactivando el parámetro `spark.sql.autoBroadcastJoinThreshold`, pero el entorno serverless de Databricks Free Edition no permite modificarlo en tiempo de ejecución. Optamos entonces por indicarle a Spark la estrategia mediante un *hint*: `merge` obliga a resolver el cruce con `SortMergeJoin`, la estrategia genérica que reparte ambas tablas entre las máquinas. La sección 4 parte de esta misma consulta y cambia únicamente esa indicación, de manera que la diferencia medida sea atribuible a la técnica y no a otro cambio.

In [0]:
# Función de medición — úsenla tal cual, no la modifiquen
def medir(descripcion, funcion, repeticiones=3):
    """Calienta, repite y reporta la mediana. Las tres mediciones quedan visibles."""
    funcion()                                   # calentamiento, se descarta
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)

    mediana = statistics.median(tiempos)
    print(f"{descripcion}")
    print(f"   mediciones : {', '.join(f'{t:.2f}s' for t in tiempos)}")
    print(f"   MEDIANA    : {mediana:.2f}s")
    print(f"   variación  : {max(tiempos)-min(tiempos):.2f}s\n")
    return mediana

In [0]:
# --- Su consulta ANTES de optimizar

clickstream = spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_plata")
bookings    = spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_plata")

# hint("merge") fuerza la estrategia genérica SortMergeJoin, que es nuestro caso base
consulta_original = (clickstream.alias("c")
    .join(bookings.alias("b").hint("merge"), F.col("c.user_id") == F.col("b.user_id"))
    .groupBy(F.col("c.device"), F.col("c.referrer"))
    .agg(
        F.count("*").alias("eventos_cruzados"),
        F.countDistinct(F.col("c.user_id")).alias("usuarios_distintos"),
        F.sum(F.when(F.col("b.status") == "confirmed", 1).otherwise(0)).alias("reservas_confirmadas")
    ))

print("Consulta definida. Spark aún no la ejecuta: falta una acción.")

**Interpretación**

La consulta cruza cada evento de navegación con las reservas del mismo usuario y agrupa el resultado por dispositivo y canal de origen, contando los eventos que produce el cruce, los usuarios distintos y las reservas confirmadas. Termina en una agregación de forma intencional: la función `medir()` ejecuta la consulta con `.collect()`, de modo que lo que viaja al nodo controlador debe ser el resultado agregado unas pocas filas y no las filas intermedias del cruce.

El `hint("merge")` cumple una función metodológica. Spark, con su configuración por defecto, detecta que `bookings_plata` es lo bastante pequeña para difundirla a cada máquina y aplica esa mejora por su cuenta, de modo que no existiría un caso base contra el cual comparar. El laboratorio guiado resuelve esto desactivando el parámetro `spark.sql.autoBroadcastJoinThreshold`, pero el entorno serverless de Databricks Free Edition no permite modificar ese parámetro en tiempo de ejecución. Optamos entonces por declarar la estrategia de forma explícita: `merge` le indica a Spark que resuelva el cruce con `SortMergeJoin`, la estrategia genérica que reparte ambas tablas entre las máquinas. La sección 4 parte de esta misma consulta y cambia únicamente esa indicación, de manera que la diferencia medida es atribuible a la técnica y no a otra modificación.

Definir la consulta no ejecuta nada. Spark construye el plan lógico y espera una acción para resolverlo, y es ese momento previo el que aprovechamos en 3.2 para inspeccionar el plan antes de medir.

## 3.2 · El plan de ejecución inicial



In [0]:
consulta_original.explain()


**Qué vemos en el plan**

El plan se lee de abajo hacia arriba. En la base, dos `PhotonScan parquet` leen las tablas Delta `clickstream_plata` y `bookings_plata`. El propio plan muestra que Spark no lee las tablas completas: el `ReadSchema` de cada rama incluye únicamente las columnas que la consulta necesita `user_id`, `device` y `referrer` de la navegación, y `user_id` y `status` de las reservas, y el `DataFilters: [isnotnull(user_id)]` descarta de entrada las filas sin llave, porque un valor nulo nunca encuentra pareja en un cruce interno. Esa poda de columnas y filas es una optimización que el motor aplica por su cuenta y no es donde está el costo.

**El costo aparece inmediatamente después, en los dos `PhotonShuffleExchangeSink hashpartitioning(user_id, 16)`.** Un `Exchange` es movimiento de datos entre máquinas: Spark calcula el hash de `user_id` en cada fila y la envía a la partición responsable de ese rango de llaves, de modo que las filas con el mismo usuario terminen juntas y el cruce pueda resolverse localmente. Esto ocurre **sobre las dos tablas**, no sobre una: las 100.000 filas de la navegación y las 72.247 de las reservas se reparten por la red en 16 particiones antes de que el cruce empiece. A cada lado le sigue además un `PhotonSort [user_id ASC NULLS FIRST]`, porque la estrategia elegida necesita ambas entradas ordenadas para poder recorrerlas en paralelo.

El nodo `SortMergeJoin [user_id], [user_id], Inner` confirma que el *hint* se respetó y que estamos midiendo efectivamente la estrategia genérica. Junto a él aparecen dos elementos que refuerzan por qué este caso base es costoso: el `ColumnarToRow` y la advertencia explícita del apartado *Photon Explanation*, donde el motor declara que **no puede ejecutar la consulta completa porque `SortMergeJoin` es un nodo no soportado**. Esto significa que los datos, leídos en formato columnar por el motor vectorizado de Photon, deben convertirse a filas para resolver el cruce en el motor de Spark convencional. Es un costo adicional que el plan hace visible y que no depende del volumen, sino de la estrategia elegida.

Por encima del cruce está la agregación, resuelta en dos fases con un tercer `Exchange hashpartitioning(device, referrer, 16)` en medio. Ese movimiento es mucho más barato que los anteriores, porque a esa altura los datos ya vienen agregados parcialmente por cada partición y lo que viaja son resultados intermedios, no filas del cruce. La agrupación intermedia por `device`, `referrer` y `user_id` que se observa en el plan corresponde al `countDistinct` sobre `user_id`: para contar usuarios distintos, Spark debe agrupar primero por esa columna antes de consolidar el resultado final.

Dos anotaciones sobre cómo interpretar este plan. La primera es que el encabezado `AdaptiveSparkPlan isFinalPlan=false` indica que se trata del plan previo a la ejecución: la ejecución adaptativa (AQE) puede ajustar detalles en tiempo real, como el número de particiones, aunque la estrategia de cruce queda fijada por el *hint*. La segunda es que el apartado de estadísticas reporta `full` para ambas tablas, es decir, Spark conoce con precisión su tamaño; justamente por eso, sin el *hint*, habría estimado que `bookings_plata` cabe en memoria y habría elegido difundirla por su cuenta.

**En resumen:** el trabajo dominante de esta consulta está en repartir y ordenar las dos tablas completas antes del cruce, y en tener que abandonar el motor vectorizado para resolverlo. Esos son los dos puntos sobre los que interviene la optimización de la sección 4.

## 3.3 · Los tres tiempos

In [0]:
tiempo_antes = medir(
    "ANTES · SortMergeJoin (clickstream_plata × bookings_plata)",
    lambda: consulta_original.collect()
)


**Interpretación**

Las tres mediciones fueron de 1.99s, 1.09s y 1.10s, con una **mediana de 1.10s** y una variación de 0.90s entre la más lenta y la más rápida. Esa es la referencia contra la cual se compara la optimización de la sección 4.

El procedimiento explica por qué reportamos la mediana y no el promedio. La función `medir()` ejecuta primero la consulta una vez y descarta ese tiempo, porque la primera ejecución paga costos que no se repiten: la planeación de la consulta, la lectura de los metadatos de Delta y el estado inicial de las cachés de lectura. Aun así, la primera de las tres mediciones válidas (1.99s) casi duplica a las dos siguientes, que fueron prácticamente idénticas entre sí (1.09s y 1.10s). Eso indica que el efecto de calentamiento no se agota por completo en una sola ejecución, y que el comportamiento estable de la consulta se alcanza a partir de la segunda. El promedio de las tres habría sido de 1.39s, un 27% por encima de la mediana, arrastrado justamente por ese primer valor; la mediana, en cambio, coincide con el comportamiento que la consulta repite.

La variación de 0.90s merece una lectura aparte, porque fija el umbral de lo que podemos afirmar. Equivale al 82% de la mediana, de modo que **cualquier diferencia inferior a ese orden de magnitud no sería distinguible del ruido normal del entorno**. Es un límite propio de trabajar con un volumen moderado 172.247 filas entre ambas tablas sobre cómputo compartido: la consulta no dura lo suficiente como para que las diferencias de rendimiento se impongan sobre la variabilidad de la plataforma. Este dato es el que permite interpretar con rigor el resultado de la sección 4: una mejora deberá ser claramente superior a esa variación para poder atribuirse a la técnica aplicada y no a la fluctuación entre ejecuciones.

# 4 · Optimización · 20 puntos

## 4.1 · Qué técnica aplicamos y por qué

*Una sola técnica, bien elegida. Expliquen por qué esa y no otra, en función de lo que
vieron en el plan inicial.*

Aplicamos **broadcast join**, forzándolo con `hint("broadcast")` sobre `bookings_plata`.

La elección no es arbitraria: se desprende directamente de lo que el plan de la sección 3.2 mostró. El costo del caso base tenía dos causas concretas. La primera eran los dos `Exchange hashpartitioning` que movían por la red **las dos** tablas completas —100.000 filas de navegación y 72.247 de reservas— antes de poder cruzarlas. La segunda era que `SortMergeJoin` no es un nodo soportado por Photon, así que Spark tenía que convertir los datos de columnar a fila (`ColumnarToRow`) para resolver el cruce fuera del motor vectorizado.

`bookings_plata` es precisamente el tipo de tabla para la que existe el broadcast join: es lo bastante pequeña para copiarla completa a cada executor. Al hacerlo, se elimina el `Exchange` de la tabla grande (`clickstream_plata` ya no necesita repartirse por la red, se procesa donde está), desaparece el ordenamiento previo que exigía el `SortMergeJoin`, y `BroadcastHashJoin` sí es un operador nativo de Photon, así que también se evita la conversión a fila. No elegimos otra técnica (como el *bucketing* o particionar de antemano las tablas) porque esas exigen rediseñar cómo se almacenan los datos; el broadcast resuelve el mismo problema con un cambio de una sola línea sobre la consulta ya existente, que es justo la comparación limpia que pide esta sección.

Esta es, de hecho, la misma decisión que Spark habría tomado automáticamente si no se lo hubiéramos impedido con el `hint("merge")` en la sección 3 — aquí simplemente la hacemos explícita para poder medirla y explicarla.

In [0]:
# ---  consulta DESPUÉS de optimizar
clickstream = spark.table(f"{CATALOGO}.{ESQUEMA}.clickstream_plata")
bookings    = spark.table(f"{CATALOGO}.{ESQUEMA}.bookings_plata")

# hint("broadcast") fuerza BroadcastHashJoin: bookings_plata se copia completa
# a cada executor, evitando repartir clickstream_plata por la red
consulta_optimizada = (clickstream.alias("c")
    .join(bookings.alias("b").hint("broadcast"), F.col("c.user_id") == F.col("b.user_id"))
    .groupBy(F.col("c.device"), F.col("c.referrer"))
    .agg(
        F.count("*").alias("eventos_cruzados"),
        F.countDistinct(F.col("c.user_id")).alias("usuarios_distintos"),
        F.sum(F.when(F.col("b.status") == "confirmed", 1).otherwise(0)).alias("reservas_confirmadas")
    ))

print("Consulta definida. Spark aún no la ejecuta: falta una acción.")

## 4.2 · El plan de ejecución resultante

In [0]:
consulta_optimizada.explain()


## 4.3 · Qué cambió entre los dos planes

*Comparen los dos planes de forma explícita. No basta con poner los dos números: hay que
decir qué operación desapareció o se redujo y por qué eso baja el costo.*

| | Antes | Después |
|---|---|---|
| Estrategia de cruce | `SortMergeJoin`: ambas tablas se ordenan por `user_id` y se recorren en paralelo buscando coincidencias | `BroadcastHashJoin`: `bookings_plata` se copia completa a cada executor y se construye una tabla hash en memoria; `clickstream_plata` se recorre una sola vez contra esa tabla |
| Movimiento de datos | Dos `Exchange hashpartitioning(user_id, 16)`: las 100.000 filas de `clickstream_plata` y las 72.247 de `bookings_plata` viajan por la red antes del cruce | Un solo `BroadcastExchange`: únicamente `bookings_plata` (la tabla pequeña) se envía, y se envía una vez a cada executor, no particionada por llave. `clickstream_plata` no se mueve |
| Ordenamiento | Dos `PhotonSort [user_id ASC]`, uno por cada tabla, requerido porque `SortMergeJoin` necesita ambas entradas ordenadas | Ninguno. `BroadcastHashJoin` resuelve el cruce con una tabla hash en memoria, que no requiere que los datos lleguen ordenados |

El punto que conecta las tres filas es que `BroadcastHashJoin` sí es un operador nativo de Photon: al eliminarse el `ColumnarToRow` que forzaba el `SortMergeJoin`, la consulta se resuelve de principio a fin en el motor vectorizado, no solo se evita el shuffle de la tabla grande.

## 4.4 · Los tiempos, comparados

In [0]:
tiempo_despues = medir("DESPUÉS · BroadcastHashJoin (clickstream_plata × bookings_plata)", lambda: consulta_optimizada.collect())

mejora = (tiempo_antes - tiempo_despues) / tiempo_antes * 100
print("═" * 48)
print(f"{'Antes':<24}{tiempo_antes:>8.2f}s")
print(f"{'Después':<24}{tiempo_despues:>8.2f}s")
print(f"{'Mejora':<24}{mejora:>8.1f}%")
print("═" * 48)

**Conclusión de la optimización:**

Antes, el cruce forzaba `SortMergeJoin`, repartiendo por la red las dos tablas completas y ordenándolas, con salida forzada del motor vectorizado de Photon. Después, con `broadcast` sobre `bookings_plata`, el plan cambió exactamente como se esperaba: desapareció el shuffle de la tabla grande y el ordenamiento de ambas, y el cruce se resolvió con `BroadcastHashJoin`, nativo de Photon.

El tiempo, sin embargo, no mostró una mejora consistente. Ejecutamos la comparación completa dos veces: la primera dio -6.3% (1.17s → 1.24s), la segunda +17.2% (1.29s → 1.07s). Que el signo del resultado cambie entre corridas de la misma consulta, sin ninguna modificación de código entre medio, es la evidencia más clara de que el efecto de esta optimización, en este volumen de datos (172.247 filas entre ambas tablas), es del mismo orden de magnitud que la variabilidad propia del entorno serverless compartido — no algo atribuible con confianza a la estrategia de cruce en sí.

Esto no invalida la optimización: el plan de ejecución sí mejoró de forma real y verificable en la sección 4.3 (menos movimiento de datos, sin caída a fila). Lo que muestra es que, a este volumen, esa mejora estructural en el plan no se traduce todavía en una diferencia de tiempo que se pueda medir con confianza — es exactamente el tipo de límite que desarrollamos en la sección 5: el beneficio del broadcast se vuelve evidente cuando el shuffle que evita empieza a doler, y en este caso el shuffle nunca llegó a doler lo suficiente.

# 5 · Límites de la optimización · 10 puntos

El broadcast join deja de servir, y empieza a empeorar el resultado, cuando la tabla que se difunde deja de ser pequeña frente a la memoria disponible en cada executor.

El mecanismo es el que explica el límite: `BroadcastHashJoin` copia la tabla completa a **cada** executor del clúster, no la reparte una vez entre todos. Eso significa que el costo no depende del tamaño total del clúster sino del tamaño de una sola copia multiplicado por el número de executors. Mientras `bookings_plata` sea del orden de decenas de miles de filas, esa copia es trivial. Pero si esa misma tabla creciera a, por ejemplo, decenas de millones de reservas —un escenario perfectamente plausible para una plataforma como Wanderbricks operando durante años, no solo el volumen de prueba con el que trabajamos—, difundirla completa a cada executor dejaría de ser gratis: cada máquina tendría que reservar en memoria una copia entera de una tabla de varios gigabytes, compitiendo por esa misma memoria con el resto del procesamiento. En el mejor caso, eso ralentiza la consulta por presión de memoria; en el peor, provoca errores de *out of memory* en los executors, que es exactamente el escenario en el que `SortMergeJoin` —más lento, pero que nunca exige tener una tabla completa en memoria en un solo nodo— vuelve a ser la opción más segura.

Lo que tendría que cambiar en nuestro caso para que esto aplique no es el volumen de `clickstream_plata` (la tabla grande), sino el de `bookings_plata` (la que se difunde): si Wanderbricks acumulara reservas históricas de varios años en lugar de las 72.247 actuales, o si decidiéramos correr esta misma consulta con una tabla de reservas ya no filtrada a un catálogo de prueba sino con el historial completo de una operación real, la tabla dejaría de calificar para broadcast y forzar el *hint* sería contraproducente en vez de útil.

Vale la pena precisar además que el broadcast solo tiene sentido para cruces por igualdad exacta (`user_id == user_id`, como es nuestro caso); no resuelve cruces por rango o por similitud, y tampoco corrige el problema de sesgo de datos (*data skew*) si unos pocos valores de `user_id` concentraran una cantidad desproporcionada de eventos en la tabla grande — ese es un problema distinto, que el broadcast no ataca porque afecta al lado que nunca se difunde.

# 6 · Cierre del caso

## 6.1 · La pregunta orientadora

> Una empresa recibe datos de su portal web, de Facebook, de Instagram, de TikTok y de WhatsApp Business. **¿Bajo qué paradigma, con qué herramientas y con qué arquitectura debería analizarlos?**

**El paradigma debe ser un lakehouse**, y la razón la vivimos en este mismo proyecto. Las cinco fuentes no entregan datos con la misma forma: el portal produce eventos tabulares, las redes sociales devuelven respuestas anidadas cuyo esquema el proveedor modifica sin previo aviso, y WhatsApp Business aporta texto libre. Un modelo relacional exigiría definir el esquema antes de conocerlo y rompería la ingesta cada vez que una API cambiara; un modelo documental absorbería bien esa variabilidad, pero renunciaría a las transacciones y a los cruces SQL entre fuentes. Nuestro caso reproduce esa misma tensión a menor escala: `clickstream` trae un campo `metadata` anidado, `reviews` trae texto libre en los comentarios, y `bookings` y `payments` son estrictamente tabulares. La resolvimos conservando el dato anidado tal como llegaba en la capa bronce y aplanándolo solo en la capa plata, sin tener que elegir un único formato desde el inicio. Y comprobamos que el esquema puede crecer sobre la marcha cuando incorporamos una columna nueva con `mergeSchema` sin romper las 72.247 filas existentes: esa es exactamente la situación de una API de red social que agrega un campo de un día para otro.

**Las herramientas serían las que usamos durante el curso**: almacenamiento en un formato abierto con garantías transaccionales, Delta Lake, que en la EA1 nos permitió demostrar atomicidad, historial de versiones, viaje en el tiempo y evolución de esquema, un motor de procesamiento distribuido para las transformaciones y los cruces entre fuentes, y una capa de gobierno centralizada como Unity Catalog. Esta última no es accesoria en este escenario: cinco fuentes implican cinco responsables distintos y datos personales de por medio, de modo que los permisos diferenciados por capa y el linaje que trabajamos en la EA2 dejan de ser un ejercicio académico y pasan a ser un requisito. A eso se suma un orquestador que ejecute la ingesta de forma programada, como el Job de dos tareas encadenadas que dejamos funcionando, y una herramienta de consumo sobre la capa final para que las áreas de negocio no dependan del equipo de datos para cada consulta.

**La arquitectura sería en capas, del tipo medallón**, que es la que construimos: una capa cruda que conserva cada fuente tal como llegó indispensable cuando el esquema de origen cambia sin aviso, porque permite reprocesar sin volver a pedir los datos, una capa intermedia que limpia, tipifica y, sobre todo, **unifica la identidad del usuario entre canales**, y una capa de consumo con una tabla por pregunta de negocio. Insistimos en el punto de la identidad porque es el que subestimamos en la EA1: al intentar cruzar la navegación con las reseñas descubrimos que ambas tablas no compartían la misma población de usuarios y tuvimos que cambiar el criterio de atribución. Con cinco fuentes distintas, donde la misma persona aparece con identificadores diferentes en cada red social, ese problema no es un detalle de implementación sino el núcleo del diseño.

**Y esta decisión tiene un costo que conviene reconocer.** Adoptar una plataforma gestionada implica dependencia del proveedor, y el propio ejercicio nos lo mostró: el entorno serverless no nos permitió modificar un parámetro de configuración de Spark que el laboratorio del curso sí usaba, y tuvimos que resolver la medición por otra vía. Esa dependencia se mitiga guardando en un formato abierto como Delta, que puede leerse desde otros motores, pero no desaparece. El segundo costo es operativo: una arquitectura en capas duplica el almacenamiento, porque el mismo dato vive en bronce y en plata, y exige mantener un pipeline que alguien debe vigilar. A cambio, ofrece lo que ninguna de las alternativas daba al mismo tiempo: absorber formatos cambiantes sin perder el dato original, y consultarlos todos con SQL.

## 6.2 · Conclusiones del proyecto

**Lo que funcionó: la cadena de capas hizo visibles problemas que el dato crudo escondía.** El recorrido de la sección 2 lo demuestra con cifras. La capa bronce conservó los 321.678 registros de origen sin alterarlos, y la capa plata, al conservar una fila por identificador, reveló dos problemas de calidad que ningún conteo en bronce habría mostrado: 4.465 pagos duplicados (49.638 filas frente a 45.173 `payment_id` distintos) y, sobre todo, que las 99.793 reseñas eran en realidad 1.000 identificadores distintos repetidos cerca de cien veces. Cualquier métrica calculada sobre bronce habría multiplicado por cien el peso de cada reseña sin que nadie lo notara. La capa oro, por su parte, condensó 96.839 combinaciones de usuario, dispositivo y canal en 12 filas legibles por el equipo de marketing, con una tasa de conversión global de 11.23%, la métrica ya calculada y las columnas documentadas en Unity Catalog. También funcionó el método de medición: reportar la mediana tras un calentamiento y registrar la variación (0.90s, el 82% de la mediana en la línea base) nos dio un criterio objetivo para saber qué podíamos afirmar y qué no.

**Lo que no funcionó: la optimización cambió el plan, pero no produjo una mejora de tiempo que podamos afirmar.** El plan resultante fue el esperado: desaparecieron los dos `Exchange` que repartían las tablas, los dos `Sort` y la conversión `ColumnarToRow`, y el cruce quedó como `BroadcastHashJoin` dentro de Photon. Los tiempos, sin embargo, no lo confirmaron. Las dos comparaciones completas que ejecutamos dieron resultados de signo contrario: 1.17s contra 1.24s (−6.3%) y 1.29s contra 1.07s (+17.2%). Que la misma consulta, sin cambios de código, pase de empeorar a mejorar entre corridas indica que el efecto de la técnica es del mismo orden que la variabilidad del entorno. Dos hechos que medimos lo explican. El volumen (172.247 filas entre ambas tablas) es demasiado pequeño para que el shuffle evitado pese más que el costo de difundir la tabla. Y nuestra línea base era en parte artificial: Spark, sin el `hint("merge")`, habría difundido `bookings_plata` por su cuenta al saber que cabe en memoria, y tuvimos que forzar ese caso base porque el entorno serverless de la Free Edition no permite modificar `spark.sql.autoBroadcastJoinThreshold`. La conclusión honesta es que la técnica es correcta en su mecanismo, pero este experimento no tiene la escala para demostrarla con tiempos.

**Una limitación de la capa oro que conviene declarar.** La tasa de 11.23% mide cuántos usuarios que navegaron tienen, en algún momento, una reserva confirmada y pagada; no mide un embudo en el tiempo. En la EA1 comprobamos que `clickstream` se concentra en una ventana de segundos del 4 de septiembre de 2025, mientras que las reservas se extienden de diciembre de 2022 a julio de 2025, así que no podemos afirmar que el usuario reservó *después* de navegar. Es una asociación por usuario, útil para comparar canales entre sí, pero no una medida del tiempo de conversión.

**Qué haríamos distinto si empezáramos de nuevo.** Primero, definiríamos el protocolo de medición antes de ver resultados: correr el caso base y el optimizado de forma consecutiva, en la misma sesión y en varias rondas, en lugar de descubrir la inconsistencia después de una sola comparación. Segundo, elegiríamos una operación cuyo costo no dependa de una optimización que el motor ya aplica solo, o escalaríamos el volumen de `clickstream` para que la diferencia entre estrategias superara el ruido; medimos justo el caso donde la técnica tenía menos que ganar. Tercero, exploraríamos la coherencia de identidad y de fechas entre las fuentes antes de diseñar las métricas, porque fue lo que en la EA1 nos obligó a rediseñar el análisis. Y cuarto, incorporaríamos validaciones de conteo entre capas al pipeline automatizado: la caída de 99.793 a 1.000 reseñas la explicamos con evidencia, pero la encontramos revisando el recorrido, no porque el pipeline avisara.

# 7 · Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| **Arley Steven Franco Galeano** | Sección 2: construcción y documentación de la capa oro (tabla Delta agregada, comentarios de tabla y columnas) y el recorrido del dato con los conteos por capa. Sección 6.1: respuesta a la pregunta orientadora del curso. | Explica cómo se diseñó la capa oro para que se entienda sin explicación, y responde la pregunta orientadora. |
| **Hanna Jineth Contreras Salinas** | Sección 3: elección y justificación de la operación costosa, captura y lectura del plan de ejecución inicial, y las tres mediciones de la línea base con la función `medir()`. Además, la Sección 1 y la consolidación final del notebook. | Explica por qué ese cruce es la operación costosa, muestra el plan inicial y cómo se midió sin engañarse. |
| **Andres Felipe Grisales Vahos** | Sección 4: aplicación de la técnica de optimización, captura del plan resultante, comparación de los dos planes y nueva medición. Sección 5: límites de la técnica. | Explica por qué se eligió esa técnica y no otra, qué operación cambió en el plan y en qué condiciones dejaría de servir. |
| **Los tres** | Sección 6.2 (conclusiones), construidas en conjunto a partir de los resultados medidos. | Cada uno responde las tres preguntas obligatorias desde la parte que trabajó. |

**Uso de asistentes de IA:** 